In [1]:
# will use 5 coins
# SAND, BORA, PLA, DKA, MANA
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_finance import candlestick_ohlc
import matplotlib.dates as mdates
import pyupbit

/root/miniconda3/envs/finance/lib/python3.9/site-packages/mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [12]:
sample = pyupbit.get_ohlcv("KRW-MANA", count=10000, interval="minute15")
sample['number'] = sample.index.map(mdates.date2num)
len(sample)

10000

In [13]:
# preprocess
sample['ema5'] = sample['close'].ewm(span=5).mean()
sample['ema7'] = sample['close'].ewm(span=7).mean()
sample['ema15'] = sample['close'].ewm(span=15).mean()
sample['ema25'] = sample['close'].ewm(span=25).mean()
sample['ema60'] = sample['close'].ewm(span=60).mean()
sample['min10'] = sample['close'].rolling(window=10).min()
sample['min15'] = sample['close'].rolling(window=15).min()
sample['min20'] = sample['close'].rolling(window=20).min()
sample['min60'] = sample['close'].rolling(window=60).min()
sample.dropna(inplace=True)
sample['rtn'] = sample['close'].pct_change()
sample['acc_rtn'] = (1. + sample['rtn']).cumprod()
sample.tail()

,open,high,low,close,volume,value,number,ema5,ema7,ema15,ema25,ema60,min10,min15,min20,min60,rtn,acc_rtn
2021-11-09 20:45:00,3105.0,3140.0,3065.0,3095.0,1.134837e+06,3.514237e+09,18940.864583,3130.726637,3141.694731,3162.553757,3174.106152,3202.579649,3095.0,3095.0,3095.0,3095.0,-0.003221,3.902900
2021-11-09 21:00:00,3095.0,3105.0,3020.0,3030.0,1.987578e+06,6.096328e+09,18940.875000,3097.151091,3113.771049,3145.984538,3163.021064,3196.921300,3030.0,3030.0,3030.0,3030.0,-0.021002,3.820933
2021-11-09 21:15:00,3050.0,3095.0,3010.0,3090.0,1.541659e+06,4.693945e+09,18940.885417,3094.767394,3107.828286,3138.986471,3157.404059,3193.415683,3030.0,3030.0,3030.0,3030.0,0.019802,3.896595
2021-11-09 21:30:00,3090.0,3125.0,3085.0,3115.0,1.060340e+06,3.287090e+09,18940.895833,3101.511596,3109.621215,3135.988162,3154.142208,3190.844677,3030.0,3030.0,3030.0,3030.0,0.008091,3.928121
2021-11-09 21:45:00,3115.0,3120.0,3095.0,3095.0,2.305835e+05,7.162450e+08,18940.906250,3099.341064,3105.965911,3130.864642,3149.592808,3187.702229,3030.0,3030.0,3030.0,3030.0,-0.006421,3.902900


In [16]:
book = sample[['number', 'close']].copy()
book['rtn'] = 0.
for idx in sample.index:
    if idx == sample.index[0]:
        continue
    
    if sample.shift(1).loc[idx, 'close'] > sample.shift(1).loc[idx, 'min60']:
        book.loc[idx, 'rtn'] = sample.loc[idx, 'close']/sample.loc[idx, 'open'] - 1.

book['acc_rtn'] = (1.+book['rtn']).cumprod()

book.tail()

,number,close,rtn,acc_rtn
2021-11-09 20:45:00,18940.864583,3095.0,0.000000,1.775352
2021-11-09 21:00:00,18940.875000,3030.0,0.000000,1.775352
2021-11-09 21:15:00,18940.885417,3090.0,0.000000,1.775352
2021-11-09 21:30:00,18940.895833,3115.0,0.008091,1.789716
2021-11-09 21:45:00,18940.906250,3095.0,-0.006421,1.778225


In [17]:
# evaluate
CAGR = sample['acc_rtn'].iloc[-1]**(365*24*4/len(sample.index)) - 1.
historical_max = sample['acc_rtn'].cummax()
daily_drawdown = sample['acc_rtn']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(sample['rtn'])*np.sqrt(365*96)
Sharpe = (np.mean(sample['rtn'])/np.std(sample['rtn']))*np.sqrt(365*96)

print("==== Buy and hold ====")
print(f"CAGR: {round(CAGR*100, 2)}%")
print(f"MDD: {round(MDD*100, 2)}%")
print(f"VOL: {round(VOL*100, 2)}%")
print(f"Sharpe: {round(Sharpe*100, 2)}%")

CAGR = book['acc_rtn'].iloc[-1]**(365*24*4/len(book.index)) - 1.
historical_max = book['acc_rtn'].cummax()
daily_drawdown = book['acc_rtn']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(book['rtn'])*np.sqrt(365*96)
Sharpe = (np.mean(book['rtn'])/np.std(book['rtn']))*np.sqrt(365*96)

print("==== EMA ====")
print(f"CAGR: {round(CAGR*100, 2)}%")
print(f"MDD: {round(MDD*100, 2)}%")
print(f"VOL: {round(VOL*100, 2)}%")
print(f"Sharpe: {round(Sharpe*100, 2)}%")

==== Buy and hold ====
CAGR: 12048.44%
MDD: -42.32%
VOL: 173.35%
Sharpe: 362.86%
==== EMA ====
CAGR: 660.59%
MDD: -47.98%
VOL: 167.74%
Sharpe: 203.99%
